# VULCAN

## IMPORTING MODULES

In [ ]:
import pandas as pd
import numpy as np

## LOADING RAW CSVs

In [ ]:
complaints = pd.read_csv('complaints.csv')
customers = pd.read_csv('customers.csv', dtype={'phone': str})
service = pd.read_csv('service_records.csv')
vehicles = pd.read_csv('vehicles.csv')

print(complaints.shape, customers.shape, service.shape, vehicles.shape)

## INSPECTING THE CSVs

In [ ]:
complaints_clean = complaints.copy()
customers_clean = customers.copy()
service_clean = service.copy()
vehicles_clean = vehicles.copy()

We will use these copies to make any changes without altering the actual CSVs.

### complaints.csv

In [ ]:
complaints.head()

In [ ]:
complaints.describe()

In [ ]:
complaints.info()

In [ ]:
complaints.isna().sum()

In [ ]:
complaints['category'].value_counts()

No obvious category inconsistency was observed from the displayed values.

In [ ]:
complaints['status'].value_counts()

There are same categories that are being treated different just because they are written differently. We have to resolve this issue.

In [ ]:


complaints_clean['status'] = complaints_clean['status'].str.strip().str.capitalize()

In [ ]:
complaints_clean['status'].value_counts()

Now, we have the 4 standard status - Resolved, Open, Escalated, Closed

***Hypothesis-1***

From this data, we can make a hypothesis that the missing values in resolution_days may be missing because those complaints have not yet been resolved — meaning complaints with an unresolved status such as Open (including open) may have resolution_days = NaN.

In [ ]:
complaints_clean[complaints_clean['resolution_days'].isna()]['status'].value_counts()

The missing values contains the values from all the categories, so we reject this hypothesis.

Now lets move on to the next investigation. We want to know if every complaint ID is unique.

In [ ]:
complaints_clean['complaint_id'].duplicated().sum()

Every complaint_id is unique. So there is no problem in this. 

Lets check if the dates are correct as they are being treated as objects right now. We want to know if they are consistantly formatted.

In [ ]:
complaints_clean['complaint_date'].head(10)

The dates appear to be in a conistant format of YYYY-MM-DD, but we need to check if it is true for the whole data. 

In [ ]:
pd.to_datetime(complaints_clean['complaint_date'], errors= 'coerce').isna().sum()

The values are consistant throughout the dataset. So we convert the dates to the proper date format.

In [ ]:
complaints_clean['complaint_date'] = pd.to_datetime(complaints_clean['complaint_date'])

In [ ]:
complaints_clean.info()

The complaint_date comlumn was succesfully converted to `datetime64` from `object`.

We have cleaned the complaint_date column and have standardized the status column. So we focus on the category column now, where 8 values are missing.

The question is - *What do these 8 missing complaints look like and can we find information in other columns regarding this.*

In [ ]:
complaints_clean[complaints_clean['category'].isna()]

From this data we were not able to find any relation between category and status. Nor were we able to guess a value from the other columns. So we will label them *Uncategorized*. 

In [ ]:
complaints_clean['category'] = complaints_clean['category'].fillna('Uncategorized')

In [ ]:
complaints_clean['category'].isna().sum()

Lets move on from the category column to the main unresolved issue - the *resolution_days* column. 

In [ ]:
complaints_clean[complaints_clean['resolution_days'].isna()]

`resolution_days` contained 63 missing values. Investigation showed that these missing values were spread across various complaint status and categories. So we cannont actually inferr the values from these fields. The missing values were therefore retained rather than artificially imputed.

In [ ]:
complaints_clean.info()

The **complaints.csv** is cleaned for now. Now we will move to the customers.csv

### customers.csv

In [ ]:
customers.head

In [ ]:
customers.info()

We can see that the `email` column has values missing. Lets see the exact number of missing values.

In [ ]:
customers.isna().sum()

There are 26 missing values in the `email` column. We will inspect the missing values and try to find a relation between other columns to see if we can reasonably fill the missing values.

In [ ]:
customers_clean[customers_clean['email'].isna()]

26 customer records have missing email addresses. After examining the affected records, the missing emails could not be reliably inferred from the available columns such as name, phone number, city, or signup date. Therefore, the missing values will not be fabricated. So we move on without changing the records. 

Now we will check if all the values in the email column are actually in the right format or not. 

In [ ]:
customers_clean.head(10)

The emails look structurally consistant. The email column is cleaned now. Lets move to the `signup_date` column. 

In [ ]:
customers_clean['signup_date'].head(10)

The `signup_date` column values are consistantly in the form of YYYY-MM-DD. Lets check if its true for the whole column.

In [ ]:
pd.to_datetime(customers_clean['signup_date'], errors= 'coerce').isna().sum()

There is no unparseable value found in the investigation. So lets change the column in the proper date format.

In [ ]:
customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'])

In [ ]:
customers_clean.info()

The data type of the *signup_date* columnn was changed from `object` to `datetime64`. 

The next column we check is the `phone` column. We need to know if all the phone numbers are right or are there anomalies in there.

In [ ]:
customers_clean['phone'].astype(str).str.len().value_counts()

In [ ]:
customers_clean['phone'].duplicated().sum()

Now we know that there are  15 phone numbers that are duplicate. We will have a look at numbers that are duplicate before proceeding.

In [ ]:
customers_clean[customers_clean['phone'].duplicated(keep=False)].sort_values('phone')

Here we can see that there are many duplicate phone numbers that are caused due to duplicate records. Before deleting the duplicates, I want to see that - How many records are duplicate when we compare all the indentifying customer information?

In [ ]:
customers_clean.duplicated(subset= ['name','phone','email','city','signup_date']).sum()

In [ ]:
customers_clean[
    customers_clean['name'].str.lower().duplicated(keep=False)
].sort_values('name')

We can see that there are exact duplicates present in the data(`C9xxx` column) with different `customer_id`. 

Lets standardize the names and check for duplicates again.

In [ ]:
customers_clean['name'] = customers_clean['name'].str.strip().str.title()

In [ ]:
customers_clean.duplicated(subset=['name', 'phone', 'email', 'city', 'signup_date']).sum()

There are 15 duplicate values, we will have a look at this.

In [ ]:
customers_clean[customers_clean.duplicated(subset=['name', 'phone', 'email', 'city', 'signup_date'],keep=False)].sort_values(['name', 'phone'])

Here we can see that `C9xxx` series has duplicate values. Before we delete this series lets confirm our hypothesis

In [ ]:
customers_clean[customers_clean['customer_id'].str.startswith('C90')].sort_values(['customer_id','name'])

In [ ]:
customers_clean[customers_clean['customer_id'].str.startswith('C90')].shape[0]

Now, it is confirmed that the `C9xxx` series contains duplicate values. So we remove the series. But before that we need to check if the series is refrenced in the vehicles.csv as there is a column `customer_id` present there.  

In [ ]:
c9_id = customers_clean[customers_clean['customer_id'].str.startswith('C9')]['customer_id']

We made `c9_id` that is basically the `C9xxx` series. Now we will check if they are refrenced in any other table before deleting the `C9xxx` series. Now we check if it is in `vehicles.csv`.

In [ ]:
vehicles[vehicles['customer_id'].isin(c9_id)].shape[0]

There is 24 entries refrenced to `C9xxx` series in the `vehicles.csv`. First we will map the IDs and change them in vehicles.csv and then we will delete the series.

In [ ]:
customer_id_map = {
    'C9000': 'C0399',
    'C9001': 'C0126',
    'C9002': 'C0329',
    'C9003': 'C0340',
    'C9004': 'C0173',
    'C9005': 'C0343',
    'C9006': 'C0198',
    'C9007': 'C0292',
    'C9008': 'C0030',
    'C9009': 'C0285',
    'C9010': 'C0175',
    'C9011': 'C0373',
    'C9012': 'C0189',
    'C9013': 'C0325',
    'C9014': 'C0322'
}

In [ ]:
vehicles_clean['customer_id'] = vehicles_clean['customer_id'].replace(customer_id_map)

We replaced the `C9xxx` series with the actual IDs. Lets check it.

In [ ]:
vehicles_clean[~vehicles_clean['customer_id'].isin(customers_clean['customer_id'])]

There is no refrence present now, so we are safe to delete the series.

In [ ]:
customers_clean = customers_clean[~customers_clean['customer_id'].str.startswith('C90')]

We have deleted the `C9xxx` series. Lets confirm it.

In [ ]:
customers_clean.shape

In [ ]:
customers_clean.duplicated(subset=['name','phone','email','city','signup_date']).sum()

There are 400 rows present instead of 415 and no duplicates present. Hence we can confirm that the `C9xxx` series was removed. 

Now we move back to checking the phone numbers. We found that there are there were 15 duplicates that have been taken care of. Now that the records are gone lets recheck the records.

In [ ]:
customers_clean['phone'].astype(str).str.len().value_counts().sort_index()

In [ ]:
assert customers_clean['phone'].str.len().eq(10).all()

There are no invalid numbers present. Now we move to the `city` column.

In [ ]:
customers_clean['city'].value_counts()

The city column seem to be alright. Lets check if there are any missing values before moving forward.

In [ ]:
customers_clean['city'].isna().sum()

There is no missing value present. We now move forward to the `customer_id` column.

In [ ]:
customers_clean['customer_id'].head(20)

We can see that the IDs follow a consistent `Cxxxx` pattern. Lets check if there is any ID in the column that does not follow the pattern.

In [ ]:
customers_clean[~customers_clean['customer_id'].str.match(r'^C\d{4}$')]

Now we have cleaned the customers csv file. We will run the final test before moving forward to the next CSV.

In [ ]:
customers_clean.info()

In [ ]:
customers_clean.isna().sum()

Both the test show that there are no missing values present(apart from the missing 26 emails) and the values are of correct data type. Lets move on to the next dataset `service_records.csv`.

### service_records.csv

In [ ]:
service.info()

In [ ]:
service.isna().sum()

Here, we can clearly see that there are missing values present and some data columns also don't have the correct data type. 

We will continue the cleaning column wise starting from `service_id`. First we shall check if all the id's are unique.

In [ ]:
service['service_id'].duplicated().sum()

There are 10 rows with duplicate values present in the column. We need to fix this. First we will check the values that are duplicate. 

In [ ]:
service[service['service_id'].duplicated(keep=False)].sort_values('service_id')

Here we can clearly see that there are exact duplicates present in the file. We need to remove these. 

In [ ]:
service_clean = service_clean.drop_duplicates()

In [ ]:
service_clean.shape

The duplicates were removed. Let us verify before moving forward.

In [ ]:
service_clean.duplicated().sum()

We have confirmed that the duplicated were removed, so lets move to the next column `vehicle_id`. 

This column is a foreign key type column so duplicates will be present as a single car can have multiple service records. What we need to know is if the ID's correspond to actual vehicles. 

In [ ]:
service_clean[~service_clean['vehicle_id'].isin(vehicles['vehicle_id'])]

Every vehicle id present in service_records.csv has a corresponding vehicle in the vehicles.csv. So, we can treat `service_clean[\'vehicle_id\']` as a foreign key refferancing to `vehicles[\'vehicle_id\']`.  

Now we move to the next column - `service_date`. \
First we need to check if the dates are in correct format.

In [ ]:
service_clean['service_date'].head(20)

There is NaN present for dates that are not available. We want to check if there is are any invalid or missing dates(apart from the 40 we know are missing) present in the column.

In [ ]:
pd.to_datetime(service_clean['service_date'], errors='coerce').isna().sum()

There is no missing date present apart from the ones we know. So now we can properly convert the the column to datetime type.

In [ ]:
service_clean['service_date'] = pd.to_datetime(service_clean['service_date'])

In [ ]:
service_clean['service_date'].info

The column was successfully converted to datetime.

Now we move to `service_type`.\
We will check of there are inconsistent spellings or capitalization.

In [ ]:
service_clean['service_type'].value_counts()


No anomaly is found in this column so we move forward to the `technician` column. This column has 236 missing values. 

In [ ]:
service_clean['technician'].value_counts()

The column looks fine, there are no issues except the 236 missing values. We need to check why they are missing, if they are concentrated around certain type of services.

In [ ]:
service_clean[service_clean['technician'].isna()]['service_type'].value_counts()

The missing values are distributed among the services and no single service stands out. So, we cannot say that the missing technicians are associated with the service_type column.\ 
Maybe the missing techinicians are associated with the particular `days_taken` values. 

In [ ]:
service_clean[service_clean['technician'].isna()]['days_taken'].value_counts().sort_index()

This does not give a particular pattern either. Lets check if it is related to the service cost.

In [ ]:
service_clean[service_clean['technician'].isna()]['cost_inr'].describe()

Here, we can see that the minimum cost is -1 which is not possible which is a issue we will come back to later. But apart from that we could not find a pattern in this either so we can not construct the missing values ourselfs. We leave them as it is. We will fill them unknown in future if needed.

Now we switch focus on the next column - `cost_inr`. \
We had already discovered the -1 value, so we will check for that. 

In [ ]:
service_clean[service_clean['cost_inr'] <=0 ]

There are 6 affected rows in the dataset. There is no specific service or technichian that has the wrong values. Now this `-1` should be changed to `NaN` as the values are invalid. We will not fill them with median yet, we shall look at the rest of the dataset and then decide if it will be correct to fill these values with median. But before we do it lets check if there are any other suspicious values in the column.

In [ ]:
service_clean['cost_inr'].describe()

The maximum value is high but not too high. The rest doesn't look suspicious. So we shall change `-1` to `NaN`.

In [ ]:
service_clean['cost_inr'] = service_clean['cost_inr'].replace(-1, np.nan)

In [ ]:
service_clean['cost_inr'].isna().sum()

Now lets move to our next column - `days_taken`.

Lets first see what the values look like.

In [ ]:
service_clean['days_taken'].value_counts().sort_index()

This is actually a very clean column. There are no missing values and the days look fine as well. Before moving to the next csv lets check if our cleaning actually produce the desired values.

In [ ]:
service_clean.isna().sum()

Alright now lets move to the last datset - `vehicles.csv`. 


### vehicles.csv


First we will check the `vehicle_id` column. This column works as a foriegn key that is referenced in the `service_records.csv`. First we need to check if there are any duplicate id's present.

In [ ]:
vehicles_clean['vehicle_id'].duplicated().sum()

There are no duplicate id's present. Lets check if all the columns follow the `Vxxxx` patern.

In [ ]:
vehicles_clean[~vehicles_clean['vehicle_id'].str.match(r'^V\d{4}$')]

There is no id that does not follow the `Vxxxx` pattern. So we move to the next column - `customer_id`.

This is a foriegn key column that is referencing to the `customers.csv`. We had already cleaned it while cleaning that csv. But lets confirm if every customer_id in this csv exists in the customers.csv.

In [ ]:
vehicles_clean[~vehicles_clean['customer_id'].isin(customers_clean['customer_id'])]

All the Ids are present in the customers.csv. That means there is no vehicle that is refrencing to a customer that does not exist. Lets check for missing values before moving forward.

In [ ]:
vehicles_clean['customer_id'].isna().sum()

There are no missing values present. Duplicates are to be expected in this column because one customer can own more than 1 car. So this column in fine.

Now lets move to the next column - `make`

In [ ]:
vehicles_clean['make'].value_counts()

There is no spelling or casing variation present. This column is fine. 

Lets move to the next column - `model`.

In [ ]:
vehicles_clean['model'].value_counts()

Lets check if the model belongs to only one make. 

In [ ]:
vehicles_clean.groupby('model')['make'].nunique().max()

There are no extra spaces and every model belongs to a single make. Lastly, lets check that the fuel type makes sense for each model (for example models like `Nexon EV` should only be EVs).

In [ ]:
pd.crosstab(vehicles_clean['model'], vehicles_clean['fuel_type_raw'])

Now we look at the `fuel_type_raw` column. The same fuel type is written in many different ways, so lets look at the values first.

In [ ]:
vehicles_clean['fuel_type_raw'].value_counts()

There are 15 variants but they are only 4 real fuel types (EV, Petrol, Diesel, Hybrid) written with different casing and names (for example `Battery Electric`, `Gasoline`, `HEV`). We will make a mapping and create a new clean column `fuel_type`, keeping the original `fuel_type_raw` untouched.

In [ ]:
fuel_map = {
    'ev': 'EV', 'electric': 'EV', 'battery electric': 'EV',
    'petrol': 'Petrol', 'gasoline': 'Petrol',
    'diesel': 'Diesel',
    'hybrid': 'Hybrid', 'hev': 'Hybrid'
}

In [ ]:
vehicles_clean['fuel_type'] = vehicles_clean['fuel_type_raw'].str.strip().str.lower().map(fuel_map)

In [ ]:
vehicles_clean['fuel_type'].isna().sum()

In [ ]:
vehicles_clean['fuel_type'].value_counts()

Now the issue is of multiple duplicate values with just casing and label difference is gone. Now there are the standard 4 fuel types which makes the output of our previous code to check the fuel types of vehicles much cleaner. Lets run that code again.

In [ ]:
pd.crosstab(vehicles_clean['model'], vehicles_clean['fuel_type'])

The EV models (like `450X`, `S1 Pro`, `Ioniq 5`, `Nexon EV`) only have the EV fuel type and no other model is marked as EV. So the models are consistent with the standardized `fuel_type`.


Now lets move to the next column:`registration_year`.\
Lets check the data type, missing values and the range of years first.

In [ ]:
vehicles_clean['registration_year'].isna().sum()

There is no missing value present and the data type is integer.

In [ ]:
vehicles_clean['registration_year'].value_counts().sort_index()

All years are between 2020 and 2026 and there are no missing values. A vehicle is usually registered in the year it is purchased, so lets check if `registration_year` matches the year of `purchase_date`.

In [ ]:
(pd.to_datetime(vehicles_clean['purchase_date']).dt.year != vehicles_clean['registration_year']).sum()

The registration year matches the purchase year for every vehicle, so `registration_year` is clean and needs no changes. We will convert `purchase_date` to datetime separately.

Next is the `purchase_date` column. Lets look at how the values are written.

In [ ]:
vehicles_clean['purchase_date'].head(10)

The `purchase_date` values are in the form of YYYY-MM-DD. Lets check if its true for the whole column.

In [ ]:
pd.to_datetime(vehicles_clean['purchase_date'], errors= 'coerce').isna().sum()

All the dates are valid, so we can safely change the data type to datetime.

In [ ]:
vehicles_clean['purchase_date'] = pd.to_datetime(vehicles_clean['purchase_date'])

In [ ]:
vehicles_clean['purchase_date'].agg(['min', 'max'])

The data type of `purchase_date` was changed from `object` to `datetime64`. The earliest purchase is in August 2020 and the latest in June 2026, so there are no future dates.


Now lets move to the last column : `odometer_km`

In [ ]:
vehicles_clean['odometer_km'].describe()

The minimum is -999 which is not possible for an odometer reading. It is a placeholder for a bad value, same as `-1` in `cost_inr`. Lets see these rows.

In [ ]:
vehicles_clean[vehicles_clean['odometer_km'] < 0]

There are 4 rows with -999. We will replace them with NaN, because -999 is not a real reading and would badly affect any average we calculate.

In [ ]:
vehicles_clean['odometer_km'] = vehicles_clean['odometer_km'].replace(-999, np.nan)

In [ ]:
vehicles_clean['odometer_km'].isna().sum()

Now there are 16 missing values (12 original + 4 from -999). Lets see if the odometer reading depends on the fuel type or the age of the vehicle, so we can estimate the missing ones.

In [ ]:
vehicles_clean.groupby('fuel_type')['odometer_km'].median()

In [ ]:
vehicles_clean.groupby('registration_year')['odometer_km'].median()

The median odometer reading is almost the same for every fuel type and every registration year, so we could not find a pattern to estimate the missing values from. We leave them as NaN instead of filling them with a guess.

The `vehicles` table is now cleaned. Lets do a final check on all columns.

In [ ]:
vehicles_clean.info()

In [ ]:
vehicles_clean.isna().sum()

Now we do the final checks. We have already checked the foreign keys for `service_records`. The last one left is `complaints`, which also refers to `vehicles` through `vehicle_id`.

In [ ]:
complaints_clean[~complaints_clean['vehicle_id'].isin(vehicles_clean['vehicle_id'])]

There are no complaints for a vehicle that does not exist. Lets also confirm the same for service records and vehicles.

In [ ]:
service_clean[~service_clean['vehicle_id'].isin(vehicles_clean['vehicle_id'])]

In [ ]:
vehicles_clean[~vehicles_clean['customer_id'].isin(customers_clean['customer_id'])]

All the foreign keys are valid, so the tables can be joined safely. We decided earlier to fill the missing technicians with `Unassigned`, because a missing technician means the job was not assigned to anyone. The missing costs, dates, odometer readings, emails and resolution days are left as NaN, because we could not estimate them without guessing.

In [ ]:
service_clean['technician'] = service_clean['technician'].fillna('Unassigned')

### Final check and conversion

Lets look at the shape and the missing values of all four tables one last time.

In [ ]:
for name, df in [('customers', customers_clean), ('vehicles', vehicles_clean), ('service', service_clean), ('complaints', complaints_clean)]:
    print(name, df.shape)
    print(df.isna().sum()[df.isna().sum() > 0], '\n')

Finally, we save the cleaned tables. These will be used for the SQL analysis.

In [ ]:
customers_clean.to_csv('customers_clean.csv', index=False)
vehicles_clean.to_csv('vehicles_clean.csv', index=False)
service_clean.to_csv('service_clean.csv', index=False)
complaints_clean.to_csv('complaints_clean.csv', index=False)